<a href="https://colab.research.google.com/github/NicollasZoratto/Atividade-10-09-26-IA-/blob/feature-01%2F10/Lista_de_Exerc%C3%ADcios_Algoritmos_de_busca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lista de Exercícios — Busca com Bibliotecas Python
### Disciplina de Inteligência Artificial


**Duas bibliotecas serão usadas:**
- **NetworkX** — a mesma biblioteca de grafos, agora usada explicitamente para os algoritmos de busca.
Documentação: https://networkx.org/documentation/stable/reference/index.html

- **simpleai** — uma biblioteca de IA que implementa BFS, DFS, UCS, IDS e busca gulosa.
Documentação: https://simpleai.readthedocs.io/en/latest/

**Como usar:** cada exercício traz um enunciado completo, com todas as informações necessárias para resolvê-lo. Escreva sua solução na célula de código logo abaixo de cada enunciado.

## Preparando o ambiente

In [ ]:
!pip install simpleai -q --break-system-packages
import networkx as nx
from simpleai.search import SearchProblem, breadth_first, depth_first, uniform_cost, iterative_limited_depth_first, greedy, astar
print('Bibliotecas prontas.')

## Recriando os problemas de referência (Aulas 17 e 11)

In [ ]:
MAPA = [
    "##########",
    "#S...#...#",
    "#.##.#.#.#",
    "#.#..#.#.#",
    "#.#.##.#.#",
    "#.#....#.#",
    "#.####.#.#",
    "#......#E#",
    "##########",
]


def encontrar(simbolo):
    for y, linha in enumerate(MAPA):
        for x, c in enumerate(linha):
            if c == simbolo:
                return (x, y)


ESTADO_INICIAL = encontrar('S')
OBJETIVO = encontrar('E')


def livre(pos):
    x, y = pos
    if y < 0 or y >= len(MAPA) or x < 0 or x >= len(MAPA[0]):
        return False
    return MAPA[y][x] != '#'


def acoes_possiveis(estado):
    x, y = estado
    candidatos = {'Direita': (x + 1, y), 'Esquerda': (x - 1, y), 'Baixo': (x, y + 1), 'Cima': (x, y - 1)}
    return [acao for acao, pos in candidatos.items() if livre(pos)]


def aplicar_acao(estado, acao):
    x, y = estado
    if acao == 'Direita': return (x + 1, y)
    if acao == 'Esquerda': return (x - 1, y)
    if acao == 'Baixo': return (x, y + 1)
    if acao == 'Cima': return (x, y - 1)


def heuristica_manhattan(estado):
    x, y = estado
    ox, oy = OBJETIVO
    return abs(x - ox) + abs(y - oy)


# Travessia do rio (Aula 11)
ESTADO_INICIAL_RIO = ('A', 'A', 'A', 'A')
ESTADO_OBJETIVO_RIO = ('B', 'B', 'B', 'B')


def estado_valido(estado):
    fazendeiro, lobo, cabra, couve = estado
    if lobo == cabra and fazendeiro != lobo:
        return False
    if cabra == couve and fazendeiro != cabra:
        return False
    return True


def aplicar_acao_rio(estado, quem_leva):
    fazendeiro, lobo, cabra, couve = estado
    nova_margem = 'B' if fazendeiro == 'A' else 'A'
    novo_estado = [nova_margem, lobo, cabra, couve]
    if quem_leva == 'lobo' and lobo == fazendeiro:
        novo_estado[1] = nova_margem
    elif quem_leva == 'cabra' and cabra == fazendeiro:
        novo_estado[2] = nova_margem
    elif quem_leva == 'couve' and couve == fazendeiro:
        novo_estado[3] = nova_margem
    return tuple(novo_estado)


def acoes_possiveis_rio(estado):
    todas = [None, 'lobo', 'cabra', 'couve']
    return [a for a in todas if estado_valido(aplicar_acao_rio(estado, a))]


# Grafo ponderado (Aula 25)
GRAFO_PONDERADO = {
    'A': {'B': 1, 'C': 4},
    'B': {'C': 1, 'D': 5},
    'C': {'D': 1},
    'D': {},
}

print('Problemas de referência recriados: labirinto, travessia do rio, grafo ponderado.')

---
## Bloco 1 — NetworkX

Você já usou o NetworkX na Aula 15 para representar grafos e descobriu, na Aula 22, que `nx.shortest_path()` usa BFS por baixo dos panos. Agora vamos usar o NetworkX explicitamente para BFS, DFS, UCS (via Dijkstra) e, de bônus, A*.

### Exercício 1 — BFS com NetworkX

**Enunciado:** construa o labirinto de referência (a matriz `MAPA`) como um grafo dirigido `nx.DiGraph`: percorra todas as células livres do labirinto e, para cada uma, adicione uma aresta para cada célula vizinha alcançável (use as funções `acoes_possiveis()` e `aplicar_acao()` já definidas na célula de setup para descobrir os vizinhos válidos de cada estado). Em seguida, use `nx.shortest_path(grafo, origem, destino)` para encontrar o caminho entre `ESTADO_INICIAL` e `OBJETIVO`. Imprima o caminho encontrado e quantos passos ele tem (lembre-se: o número de passos é o número de arestas percorridas, ou seja, `len(caminho) - 1`).


In [ ]:
import networkx as nx

grafo = nx.DiGraph()

num_linhas = len(MAPA)
num_colunas = len(MAPA[0])

for l in range(num_linhas):
    for c in range(num_colunas):
        if MAPA[l][c] == 0:
            estado_atual = (l, c)
            for acao in acoes_possiveis(estado_atual, MAPA):
                vizinho = aplicar_acao(estado_atual, acao)
                grafo.add_edge(estado_atual, vizinho)

caminho = nx.shortest_path(grafo, source=ESTADO_INICIAL, target=OBJETIVO)
num_passos = len(caminho) - 1

print("Caminho encontrado:", caminho)
print("Número de passos:", num_passos)

### Exercício 2 — DFS com NetworkX

**Enunciado:** a partir do mesmo grafo `G` construído no Exercício 1 (ou reconstruindo-o, caso prefira), gere uma árvore de busca em profundidade com `nx.dfs_tree(G, ESTADO_INICIAL)`. Dentro dessa árvore, extraia o caminho até o `OBJETIVO` usando `nx.shortest_path()` — desta vez aplicado sobre a árvore retornada por `dfs_tree()`, e não sobre o grafo original `G`. Imprima quantos passos esse caminho tem e compare com o resultado do Exercício 1: o número de passos foi o mesmo? A DFS garante encontrar o caminho mais curto? Justifique.


In [ ]:
arvore_dfs = nx.dfs_tree(grafo, source=ESTADO_INICIAL)

caminho_dfs = nx.shortest_path(arvore_dfs, source=ESTADO_INICIAL, target=OBJETIVO)
num_passos_dfs = len(caminho_dfs) - 1

print("Caminho DFS encontrado:", caminho_dfs)
print("Número de passos (DFS):", num_passos_dfs)

### Exercício 3 — UCS com NetworkX (Dijkstra)

**Enunciado:** construa o grafo ponderado A-B-C-D (dicionário `GRAFO_PONDERADO`, já definido na célula de setup) como um `nx.DiGraph`, atribuindo o custo de cada aresta ao atributo `custo` (por exemplo, `grafo.add_edge(origem, destino, custo=valor)`). A partir desse grafo, calcule e imprima:

1. o caminho de `'A'` até `'D'` usando `nx.shortest_path()` **sem** informar peso (essa função ignora os custos e conta apenas o número de saltos);
2. o caminho de `'A'` até `'D'` usando `nx.dijkstra_path()`, informando `weight='custo'`;
3. o custo total desse segundo caminho, usando `nx.dijkstra_path_length()` (também com `weight='custo'`).

Compare os dois caminhos obtidos nos itens 1 e 2: eles são iguais? Por que a busca de custo uniforme (UCS) pode preferir um caminho com mais saltos do que a BFS?


In [ ]:
import networkx as nx

grafo_p = nx.DiGraph()

for origem, vizinhos in GRAFO_PONDERADO.items():
    for destino, custo in vizinhos.items():
        grafo_p.add_edge(origem, destino, custo=custo)

caminho_bfs = nx.shortest_path(grafo_p, source='A', target='D')

caminho_ucs = nx.dijkstra_path(grafo_p, source='A', target='D', weight='custo')
custo_total_ucs = nx.dijkstra_path_length(grafo_p, source='A', target='D', weight='custo')

print("Caminho BFS (sem peso):", caminho_bfs)
print("Caminho UCS (Dijkstra):", caminho_ucs)
print("Custo total (UCS):", custo_total_ucs)

### Exercício 4 (bônus) — A* com NetworkX

**Enunciado:** resolva o labirinto de referência com `nx.astar_path()`, usando como heurística a distância Manhattan entre dois estados (você pode escrever uma função `h_manhattan(a, b)` que retorna `abs(a[0]-b[0]) + abs(a[1]-b[1])`, ou reaproveitar `heuristica_manhattan()` já definida, adaptando-a para receber dois estados). Passe essa função ao parâmetro `heuristic` de `nx.astar_path(G, ESTADO_INICIAL, OBJETIVO, heuristic=...)`. Imprima quantos passos tem o caminho encontrado e compare com o resultado da BFS (Exercício 1): os dois algoritmos encontraram o mesmo número de passos? O A* tem garantia de encontrar o caminho ótimo quando a heurística usada é admissível — a distância Manhattan é admissível neste labirinto? Explique.


In [ ]:
import networkx as nx

def h_manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

caminho_astar = nx.astar_path(grafo, source=ESTADO_INICIAL, target=OBJETIVO, heuristic=h_manhattan)
num_passos_astar = len(caminho_astar) - 1

print("Caminho A* encontrado:", caminho_astar)
print("Número de passos (A*):", num_passos_astar)

---
## Bloco 2 — simpleai

O NetworkX não tem funções prontas para aprofundamento iterativo (IDS) nem para busca gulosa. Para esses dois, vamos usar a biblioteca **simpleai**, feita especificamente para ensinar IA.

**A vantagem pedagógica:** a classe `SearchProblem` do simpleai pede exatamente os mesmos 5 componentes que você usa desde a Aula 9:

In [ ]:
from simpleai.search import SearchProblem

# A "receita" de qualquer problema no simpleai:
#
# class MeuProblema(SearchProblem):
#     def actions(self, state):      -> Ações(s)          (Aula 9)
#     def result(self, state, action) -> Resultado(s,a)    (Aula 9)
#     def is_goal(self, state):      -> Teste de objetivo  (Aula 9)
#     def cost(self, state, action, state2):  -> Custo de caminho (Aula 9 / Aula 25)
#     def heuristic(self, state):    -> h(n)               (Aula 27)

print('Compare com a tabela da Aula 9 — são os mesmos 5 componentes, com outros nomes.')

### Exercício 5 — IDS com simpleai

**Enunciado:** formule o problema da travessia do rio (fazendeiro, lobo, cabra e couve) como uma classe `SearchProblem` do simpleai — por exemplo, `TravessiaProblem`. Implemente os métodos:

- `actions(self, state)`: retorne as ações possíveis a partir do estado, reaproveitando `acoes_possiveis_rio(state)`;
- `result(self, state, action)`: retorne o novo estado, reaproveitando `aplicar_acao_rio(state, action)`;
- `is_goal(self, state)`: retorne `True` quando `state == ESTADO_OBJETIVO_RIO`.

Em seguida, instancie o problema com `TravessiaProblem(initial_state=ESTADO_INICIAL_RIO)` e resolva-o com `iterative_limited_depth_first(problema, graph_search=True)`. A partir do resultado, extraia a sequência de estados percorridos (o atributo `.path()` do resultado retorna uma lista de pares `(ação, estado)`) e imprima essa sequência, além do número total de travessias (número de estados na sequência menos 1).


In [ ]:
from simpleai.search import SearchProblem, iterative_limited_depth_first

class TravessiaProblem(SearchProblem):
    def actions(self, state):
        return acoes_possiveis_rio(state)

    def result(self, state, action):
        return aplicar_acao_rio(state, action)

    def is_goal(self, state):
        return state == ESTADO_OBJETIVO_RIO

problema_ids = TravessiaProblem(initial_state=ESTADO_INICIAL_RIO)
resultado_ids = iterative_limited_depth_first(problema_ids, graph_search=True)

sequencia_estados = [estado for acao, estado in resultado_ids.path()]
num_travessias = len(sequencia_estados) - 1

print("Sequência de estados:", sequencia_estados)
print("Número total de travessias:", num_travessias)

### Exercício 6 — Busca gulosa com simpleai

**Enunciado:** formule o grafo-armadilha da Aula 28 como uma `SearchProblem` do simpleai — por exemplo, `ArmadilhaProblem`. O grafo tem a estrutura Início → B → Beco (sem saída) e Início → C → E → Objetivo, já representada no dicionário `GRAFO_ARMADILHA` e com os valores heurísticos já fornecidos no dicionário `H_ARMADILHA` (ambos devem ser criados por você na célula de resolução, com os mesmos valores da Aula 28: `GRAFO_ARMADILHA = {'Inicio': ['B', 'C'], 'B': ['Beco'], 'Beco': [], 'C': ['E'], 'E': ['Objetivo'], 'Objetivo': []}` e `H_ARMADILHA = {'Inicio': 10, 'B': 1, 'Beco': 4, 'C': 5, 'E': 2, 'Objetivo': 0}`). Implemente:

- `actions(self, state)`: retorne `GRAFO_ARMADILHA[state]`;
- `result(self, state, action)`: a ação já é o nome do próximo estado, então basta retornar `action`;
- `is_goal(self, state)`: retorne `True` quando `state == 'Objetivo'`;
- `heuristic(self, state)`: retorne `H_ARMADILHA[state]`.

Instancie o problema com `initial_state='Inicio'` e resolva-o com `greedy(problema, graph_search=True)`. Imprima o caminho encontrado (`resultado.path()`) e observe: o algoritmo passou pelo nó `'Beco'` antes de chegar ao objetivo? Por que a busca gulosa pode se deixar enganar por um nó com heurística baixa, mesmo que ele seja um beco sem saída?


In [ ]:
from simpleai.search import SearchProblem, greedy

GRAFO_ARMADILHA = {
    'Inicio': ['B', 'C'],
    'B': ['Beco'],
    'Beco': [],
    'C': ['E'],
    'E': ['Objetivo'],
    'Objetivo': []
}

H_ARMADILHA = {
    'Inicio': 10,
    'B': 1,
    'Beco': 4,
    'C': 5,
    'E': 2,
    'Objetivo': 0
}

class ArmadilhaProblem(SearchProblem):
    def actions(self, state):
        return GRAFO_ARMADILHA[state]

    def result(self, state, action):
        return action

    def is_goal(self, state):
        return state == 'Objetivo'

    def heuristic(self, state):
        return H_ARMADILHA[state]

problema_guloso = ArmadilhaProblem(initial_state='Inicio')
resultado_guloso = greedy(problema_guloso, graph_search=True)

caminho_guloso = [estado for acao, estado in resultado_guloso.path()]
print("Caminho encontrado:", caminho_guloso)

### Exercício 7 (bônus) — Comparando todos os algoritmos do simpleai de uma vez

**Enunciado:** formule o labirinto de referência como uma `SearchProblem` do simpleai — por exemplo, `LabirintoProblem` — implementando:

- `actions(self, state)`: reaproveitando `acoes_possiveis(state)`;
- `result(self, state, action)`: reaproveitando `aplicar_acao(state, action)`;
- `is_goal(self, state)`: retorne `True` quando `state == OBJETIVO`;
- `cost(self, state, action, state2)`: retorne `1` (cada movimento tem o mesmo custo);
- `heuristic(self, state)`: retorne a distância Manhattan entre `state` e `OBJETIVO` (reaproveitando `heuristica_manhattan(state)`).

Instancie o problema com `initial_state=ESTADO_INICIAL` e, sobre ele, execute os cinco algoritmos disponíveis no simpleai: `breadth_first`, `depth_first`, `uniform_cost`, `greedy` e `astar` (todos chamados com `graph_search=True`). Para cada algoritmo, imprima em uma tabela: o nome do algoritmo, o número de passos do caminho encontrado (`len(resultado.path()) - 1`) e o custo total (`resultado.cost`). Compare os resultados: quais algoritmos encontraram o caminho ótimo (23 passos)? Algum encontrou um caminho mais longo? Por quê?


In [ ]:
from simpleai.search import (
    SearchProblem,
    breadth_first,
    depth_first,
    uniform_cost,
    greedy,
    astar
)

class LabirintoProblem(SearchProblem):
    def actions(self, state):
        return acoes_possiveis(state, MAPA)

    def result(self, state, action):
        return aplicar_acao(state, action)

    def is_goal(self, state):
        return state == OBJETIVO

    def cost(self, state, action, state2):
        return 1

    def heuristic(self, state):
        return heuristica_manhattan(state)

problema_labirinto = LabirintoProblem(initial_state=ESTADO_INICIAL)

algoritmos = [
    ("BFS", breadth_first),
    ("DFS", depth_first),
    ("UCS", uniform_cost),
    ("Gulosa", greedy),
    ("A*", astar)
]

print(f"{'Algoritmo':<12} | {'Passos':<8} | {'Custo Total':<11}")
print("-" * 37)

for nome, alg in algoritmos:
    resultado = alg(problema_labirinto, graph_search=True)
    passos = len(resultado.path()) - 1
    custo = resultado.cost
    print(f"{nome:<12} | {passos:<8} | {custo:<11}")

---
## Desafio final (avaliado)

Escolha o problema que seu grupo formulou na Aula 17 e resolva-o usando pelo menos DOIS algoritmos diferentes de biblioteca (por exemplo, `nx.shortest_path()` e `simpleai.search.astar()`).

**Entregue:**
1. O código formulando seu problema como `nx.DiGraph` OU como `SearchProblem` do simpleai (à sua escolha).
2. O resultado de cada algoritmo escolhido (caminho e custo/número de passos).
3. Um parágrafo comparando os dois resultados: eles bateram? Se não, por quê?

In [ ]:
from simpleai.search import SearchProblem, breadth_first, astar

class Puzzle2x2Problem(SearchProblem):
    def actions(self, state):
        pos_zero = state.index(0)
        linha, coluna = pos_zero // 2, pos_zero % 2
        movimentos = []

        if linha > 0: movimentos.append((-1, 0))  # Cima
        if linha < 1: movimentos.append((1, 0))   # Baixo
        if coluna > 0: movimentos.append((0, -1)) # Esquerda
        if coluna < 1: movimentos.append((0, 1))  # Direita

        return movimentos

    def result(self, state, action):
        pos_zero = state.index(0)
        linha, coluna = pos_zero // 2, pos_zero % 2
        nova_linha, nova_coluna = linha + action[0], coluna + action[1]
        nova_pos_zero = nova_linha * 2 + nova_coluna

        novo_estado = list(state)
        novo_estado[pos_zero], novo_estado[nova_pos_zero] = novo_estado[nova_pos_zero], novo_estado[pos_zero]
        return tuple(novo_estado)

    def is_goal(self, state):
        return state == (1, 2, 3, 0)

    def cost(self, state, action, state2):
        return 1

    def heuristic(self, state):
        distancia = 0
        objetivo = {1: (0, 0), 2: (0, 1), 3: (1, 0), 0: (1, 1)}
        for idx, val in enumerate(state):
            if val != 0:
                l_atual, c_atual = idx // 2, idx % 2
                l_obj, c_obj = objetivo[val]
                distancia += abs(l_atual - l_obj) + abs(c_atual - c_obj)
        return distancia

estado_inicial_puzzle = (0, 1, 3, 2)
problema_puzzle = Puzzle2x2Problem(initial_state=estado_inicial_puzzle)

resultado_bfs = breadth_first(problema_puzzle, graph_search=True)
caminho_bfs = [estado for acao, estado in resultado_bfs.path()]
passos_bfs = len(caminho_bfs) - 1

resultado_astar = astar(problema_puzzle, graph_search=True)
caminho_astar = [estado for acao, estado in resultado_astar.path()]
passos_astar = len(caminho_astar) - 1

print("--- RESULTADO BFS ---")
print("Caminho BFS:", caminho_bfs)
print("Passos BFS:", passos_bfs)
print("\n--- RESULTADO A* ---")
print("Caminho A*:", caminho_astar)
print("Passos A*:", passos_astar)